# 05 - Preference filtering + travel time

Demonstrates `reasoning/preference_filter.py`: "find me a < POI type >, with
< amenities >, ranked by how fast I can actually get there" - adding the
category/amenity data from KG Modelling with the GTFS travel-time reasoning.

## Setup

In [1]:
import sys
sys.path.insert(0, "..")

from rdflib import Graph
from reasoning.gtfs_routing import GtfsRouter
from reasoning.preference_filter import find_pois

g = Graph()
g.parse("../kg/vienna_mobility_kg.ttl", format="turtle")
router = GtfsRouter(date="20260815")
print(f"KG: {len(g)} triples, GTFS router ready")

KG: 80485 triples, GTFS router ready


## 1. Dog-friendly parks - the query that came back empty before

In the original (0-transfer) version of this notebook, this exact query (from
Albertinaplatz) returned zero reachable results out of 10 candidates shown!

In [2]:
origin_lon, origin_lat = 16.368378925180206, 48.204590314427854  # Albertinaplatz area

results = find_pois(g, router, origin_lon, origin_lat, poi_classes=["Park"],
                     required_amenities=["Dogs allowed"], top_n=10)
n_reachable = sum(1 for r in results if r["reachable"])
print(f"{n_reachable} of {len(results)} dog-friendly parks reachable within 2 transfers\n")
for r in results:
    status = f"{r['travel_time_min']} min ({r['num_transfers']} transfers)" if r["reachable"] else "no connection"
    print(f"  {r['name']:30} {status}")

10 of 10 dog-friendly parks reachable within 2 transfers

  Resselpark                     7.3 min (0 transfers)
  Modenapark                     11.6 min (0 transfers)
  GA Linke Bahngasse             12.3 min (0 transfers)
  PA Franz-Josefs-Kai            14.0 min (0 transfers)
  Venediger-Au-Park              14.3 min (0 transfers)
  GA Obere Donaustraße           14.7 min (0 transfers)
  Stefan-Weber-Park              16.8 min (0 transfers)
  Franziska-Löw-Park             16.9 min (0 transfers)
  Bock-Park                      16.9 min (0 transfers)
  Votivpark                      17.1 min (0 transfers)


## 2. Museum to a park with amenity "trampoline"

In [3]:
q = """
PREFIX schema: <https://schema.org/>
PREFIX geo: <http://www.w3.org/2003/01/geo/wgs84_pos#>
SELECT ?lon ?lat WHERE {
    ?m a schema:Museum ; schema:name "Sammlung alter Musikinstrumente" ; geo:long ?lon ; geo:lat ?lat .
}
"""
m_lon, m_lat = [(float(r.lon), float(r.lat)) for r in g.query(q)][0]

parks = find_pois(g, router, m_lon, m_lat, poi_classes=["Park"], top_n=10)
print("Parks, ranked by travel time:")
for r in parks:
    status = f"{r['travel_time_min']} min ({r['num_transfers']} transfers)" if r["reachable"] else "no connection"
    print(f"  {r['name']:30} {status}")

trampolines = find_pois(g, router, m_lon, m_lat, poi_classes=["PlaygroundArea"],
                         required_amenities=["Trampolin"], top_n=10)
n_reach = sum(1 for r in trampolines if r["reachable"])
print(f"\nPlaygrounds with a trampoline: {n_reach} of {len(trampolines)} reachable")
for r in trampolines[:5]:
    status = f"{r['travel_time_min']} min ({r['num_transfers']} transfers)" if r["reachable"] else "no connection"
    print(f"  {r['name']:30} {status}")

Parks, ranked by travel time:
  GA Helmut-Zilk-Platz           4.0 min (0 transfers)
  Schillerpark                   4.9 min (0 transfers)
  Heldenplatz (Bundesgärten)     5.5 min (0 transfers)
  GA Minoritenplatz              5.7 min (0 transfers)
  Grete-Rehor-Park               5.9 min (0 transfers)
  PA Secession                   6.6 min (0 transfers)
  Girardipark                    7.2 min (0 transfers)
  Weghuberpark                   7.3 min (0 transfers)
  Esperantopark                  7.4 min (0 transfers)
  GA Augustinplatz               7.4 min (0 transfers)

Playgrounds with a trampoline: 10 of 10 reachable
  Vally-Wieselthier-Park         9.2 min (0 transfers)
  Czapkapark                     14.3 min (0 transfers)
  Andreaspark                    14.6 min (0 transfers)
  Humboldtpark                   15.9 min (0 transfers)
  Vilma-Steindling-Promenade     16.1 min (0 transfers)


## 3. A hard time cutoff example

In [4]:
museums_30min = find_pois(g, router, origin_lon, origin_lat, poi_classes=["Museum"],
                           max_travel_time_min=30, top_n=15)
print(f"{len(museums_30min)} museums within 30 min:")
for r in museums_30min:
    print(f"  {r['name']:35} {r['travel_time_min']} min ({r['num_transfers']} transfers)")

15 museums within 30 min:
  Theatermuseum                       1.7 min (0 transfers)
  Albertina                           1.9 min (0 transfers)
  Österreichisches Filmmuseum         1.9 min (0 transfers)
  Liebesnest Albertina                2.0 min (0 transfers)
  Glasmuseum Lobmeyr                  2.3 min (0 transfers)
  Heidi Horten Collection             2.9 min (0 transfers)
  Papyrusmuseum der Österreichischen Nationalbibliothek 3.7 min (0 transfers)
  Kaiserliche Schatzkammer            3.7 min (0 transfers)
  Jüdisches Museum Wien               3.7 min (0 transfers)
  Grillparzerhaus/Österreichisches Staatsarchiv 3.8 min (0 transfers)
  Literaturmuseum der Österreichischen Nationalbibliothek 3.8 min (0 transfers)
  Mythos Mozart                       4.0 min (0 transfers)
  Haus der Geschichte Österreich      4.0 min (0 transfers)
  Ephesos-Museum                      4.1 min (0 transfers)
  Sammlung alter Musikinstrumente     4.1 min (0 transfers)


## 4. Some more random queries, to show that the preference filtering works for any category/amenity combination.

In [ ]:
m_lon, m_lat = 16.347857505922963, 48.18243376577881 # Something at Margartengürtel 104

Libraries = find_pois(g, router, m_lon, m_lat, poi_classes=["Library"], top_n=100)
print("Libraries, ranked by travel time:")
for r in Libraries:
    status = f"{r['travel_time_min']} min ({r['num_transfers']} transfers)" if r["reachable"] else "no connection"
    print(f"  {r['name']:30} {status}")


Libraries, ranked by travel time:
  Bücherei Margareten            10.2 min (0 transfers)
  Bücherei Neues Landgut         13.3 min (0 transfers)
  Bücherei Philadelphiabrücke    13.7 min (0 transfers)
  Bücherei Mariahilf             17.5 min (0 transfers)
  Bücherei Fasanviertel          18.3 min (0 transfers)
  Hauptbücherei am Gürtel        19.3 min (0 transfers)
  Bücherei Wieden                20.2 min (0 transfers)
  Bücherei am Schöpfwerk         22.5 min (0 transfers)
  Bücherei Schwendermarkt        22.6 min (0 transfers)
  Bücherei Alt-Erlaa             23.9 min (0 transfers)
  Bibliothekspädagogisches Zentrum 24.7 min (0 transfers)
  Bücherei Zirkusgasse           25.4 min (0 transfers)
  Bücherei Erdbergstraße         27.7 min (0 transfers)
  Kinderbücherei der Weltsprachen 30.9 min (0 transfers)
  Bücherei Weimarer Straße       31.3 min (0 transfers)
  Bücherei Alsergrund            32.4 min (0 transfers)
  Bücherei Hernals               32.8 min (0 transfers)
  Bücherei 

## Findings

**Performance matters (well), not just correctness.** 
The intital naive way to add transfers - call a richer `estimate_travel_time()` 
per candidate - made `find_pois()` unusably slow for large categories (Parks alone has 1,051
candidates). Restructuring around `reachable_from()` once + `travel_time_to()`
per candidate turned the computation time into "a few seconds,"
independent of how much richer the underlying search got.

**Open question, still unresolved:** 
What should the Service Layer do with the small remaining set of truly unreachable POIs (still possible even with 2 transfers)?